<a href="https://colab.research.google.com/github/Sarah-0405/Cold_Spots_Bayern/blob/main/KNN_M%C3%BCnchen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

KNN-Matrix für München wurde extra berechnet, aufgrund von zu wenig Arbeitsspeicher

Input Daten:

Drive Ordner: **Puffergebiete**: enthalten die LST-Daten für alle Pixel innerhalb des Puffergebiets (300m um die bewohnten Gebiete herum)



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


1. Daten aus Drive Ordner Puffergebiete laden (joined_350mpuffer_um_Gitterzelle_mit_lst_Aschaffenburg.gpkg,...)


In [ ]:
import os

# Define the path to the "Puffergebiete" folder in Google Drive
drive_folder_path = "/content/drive/MyDrive/Cold Spots Bayern/Puffergebiete"

# Initialize dictionaries to store file paths categorized by city and time period
gpkg_files_2024 = {}
gpkg_files_2019_2024 = {}

# Check if the folder exists
if os.path.exists(drive_folder_path):
    # List all files in the folder
    for filename in os.listdir(drive_folder_path):
        # Check if the file is a GeoPackage file
        if filename.endswith(".gpkg"):
            file_path = os.path.join(drive_folder_path, filename)

            # Extract city name and determine time period from filename
            # Assuming the filename format is "joined_350mpuffer_um_Gitterzelle_mit_lst_[cityname].gpkg" or "joined_350mpuffer_um_Gitterzelle_mit_lst_2019_2024[cityname].gpkg"
            parts = filename.replace(".gpkg", "").split("_")
            if "2019" in parts:
                # This is a 2019-2024 file
                # The city name is the last part of the filename
                city_name = parts[-1].replace("2024", "") # Remove 2024 if present at the end
                gpkg_files_2019_2024[city_name] = file_path
            else:
                # This is a 2024 file
                # The city name is the last part of the filename
                city_name = parts[-1]
                gpkg_files_2024[city_name] = file_path

    # Print the categorized files
    print("GeoPackage files for 2024:")
    for city, path in gpkg_files_2024.items():
        print(f"  {city}: {path}")

    print("\nGeoPackage files for 2019-2024:")
    for city, path in gpkg_files_2019_2024.items():
        print(f"  {city}: {path}")

else:
    print(f"The folder '{drive_folder_path}' was not found. Please check the path.")

GeoPackage files for 2024:
  Passau: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_Passau.gpkg
  Nuremberg: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_Nuremberg.gpkg
  Munich: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_Munich.gpkg
  Landshut: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_Landshut.gpkg
  Ingolstadt: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_Ingolstadt.gpkg
  Erlangen: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_Erlangen.gpkg
  Bayreuth: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_Bayreuth.gpkg
  Schweinfurt: /content/drive/MyDrive/Cold Spots Bayern/Puffergebiete/joined_350mpuffer_um_Gitterzelle_mit_lst_Schweinfur

2. KNN Analyse von München beginnend (kann dann einfach abgebrochen werden wenn München fertig ist)

In [ ]:
!pip install esda pysal

In [ ]:
import esda
from esda.getisord import G_Local
import gc
import libpysal
import libpysal.io # Import libpysal.io for saving weights
import pandas as pd
import os
import geopandas as gpd

# Define the directory to save the weights matrices
weights_save_dir = "/content/drive/MyDrive/Cold Spots Bayern/Pufferzonen_KNN_Weights"
if not os.path.exists(weights_save_dir):
    os.makedirs(weights_save_dir)
    print(f"Created directory: {weights_save_dir}")
else:
    print(f"Directory already exists: {weights_save_dir}")

# --- Step 1: Calculate and save KNN weights matrices ---

print("\n--- Step 1: Calculating and saving KNN weights matrices ---")

# Define the starting city for KNN calculation
# Enter the name of the city from which you want to start processing, or leave it empty to start from the beginning.
start_city_knn = "Munich"

start_processing_knn = False if start_city_knn else True


# We only need to calculate the KNN matrix once per city, based on the geometry of the grid cells.
# We can use the GeoDataFrames from either time period for this, as the geometry is the same.
# Let's use the 2024 files as they are likely loaded and available via gpkg_files_2024.

processed_cities_for_weights = set() # Keep track of cities for which we've calculated weights

# Iterate through one of the file dictionaries to get city names and paths
for city_name, file_path in gpkg_files_2024.items(): # Using 2024 files to get geometries
    if not start_processing_knn:
        if city_name == start_city_knn:
            start_processing_knn = True
            print(f"  Starting KNN processing from city: {city_name}")
        else:
            print(f"  Skipping KNN processing for city: {city_name}")
            continue # Skip to the next city

    if city_name in processed_cities_for_weights:
        print(f"  Weights already calculated for {city_name}. Skipping.")
        continue

    print(f"  Calculating KNN weights for {city_name}")
    try:
        # Load the GeoPackage file into a GeoDataFrame (only geometry is needed for weights)
        gdf = gpd.read_file(file_path)

        # Ensure the GeoDataFrame has a valid projection for distance calculation
        if gdf.crs is None:
             print("  Warning: GeoDataFrame has no CRS. Assigning EPSG:3035 for calculation.")
             gdf_projected = gdf.copy()
             gdf_projected.crs = "EPSG:3035"
        elif gdf.crs.is_geographic:
             print("  Warning: GeoDataFrame is in a geographic CRS. Reprojecting to EPSG:3035 for accurate KNN calculation.")
             gdf_projected = gdf.to_crs(epsg=3035)
        else:
             gdf_projected = gdf

        # Handle cases where there are too few points for k=8
        if len(gdf_projected) <= 8:
            print(f"  Warning: Insufficient points ({len(gdf_projected)}) for KNN with k=8. Skipping weights calculation for {city_name}.")
            # Add to processed set to avoid re-attempting
            processed_cities_for_weights.add(city_name)
            continue

        # Calculate KNN weights matrix
        wq = libpysal.weights.KNN.from_dataframe(gdf_projected, k=8)
        # No transformation needed for saving, will transform before G_Local calculation

        # Define the path to save the weights matrix
        weights_file_path = os.path.join(weights_save_dir, f"knn_weights_{city_name.replace(' ', '_')}.gal")

        # Save the weights matrix explicitly opening and closing the file
        wfile = libpysal.io.open(weights_file_path, 'w')
        try:
            wfile.write(wq)
        finally:
            wfile.close()


        print(f"  KNN weights matrix saved for {city_name}: {weights_file_path}")

        processed_cities_for_weights.add(city_name)

    except Exception as e:
        print(f"  Error calculating or saving weights for {city_name}: {e}")
    finally:
        # Explicitly delete the GeoDataFrame and weights matrix
        if 'gdf' in locals():
            del gdf
        if 'gdf_projected' in locals():
            del gdf_projected
        if 'wq' in locals():
            del wq
        gc.collect()
        print(f"  Finished weights processing for {city_name}. Memory freed.")

print("\n--- Step 1 completed. KNN weights matrices saved. ---")


nochmal analog nur mit 2019-2024 Daten

In [ ]:
import esda
from esda.getisord import G_Local
import gc
import libpysal
import libpysal.io # Import libpysal.io for saving weights
import pandas as pd
import os
import geopandas as gpd

# Define the directory to save the weights matrices
output_dir = "/content/drive/MyDrive/Cold Spots Bayern/Pufferzonen_KNN_Weights" # oder zb output_KNN
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")
else:
    print(f"Directory already exists: {output_dir}")

# --- Step 1: Calculate and save KNN weights matrices ---

print("\n--- Step 1: Calculating and saving KNN weights matrices ---")

# Define the starting city for KNN calculation
# Enter the name of the city from which you want to start processing, or leave it empty to start from the beginning.
start_city_knn = "Munich"

start_processing_knn = False if start_city_knn else True


# We only need to calculate the KNN matrix once per city, based on the geometry of the grid cells.
# We can use the GeoDataFrames from either time period for this, as the geometry is the same.
# Let's use the 2024 files as they are likely loaded and available via gpkg_files_2024.

processed_cities_for_weights = set() # Keep track of cities for which we've calculated weights

# Iterate through one of the file dictionaries to get city names and paths
for city_name, file_path in gpkg_files_2019_2024.items(): # Using 2024 files to get geometries
    if not start_processing_knn:
        if city_name == start_city_knn:
            start_processing_knn = True
            print(f"  Starting KNN processing from city: {city_name}")
        else:
            print(f"  Skipping KNN processing for city: {city_name}")
            continue # Skip to the next city

    if city_name in processed_cities_for_weights:
        print(f"  Weights already calculated for {city_name}. Skipping.")
        continue

    print(f"  Calculating KNN weights for {city_name}")
    try:
        # Load the GeoPackage file into a GeoDataFrame (only geometry is needed for weights)
        gdf = gpd.read_file(file_path)

        # Ensure the GeoDataFrame has a valid projection for distance calculation
        if gdf.crs is None:
             print("  Warning: GeoDataFrame has no CRS. Assigning EPSG:3035 for calculation.")
             gdf_projected = gdf.copy()
             gdf_projected.crs = "EPSG:3035"
        elif gdf.crs.is_geographic:
             print("  Warning: GeoDataFrame is in a geographic CRS. Reprojecting to EPSG:3035 for accurate KNN calculation.")
             gdf_projected = gdf.to_crs(epsg=3035)
        else:
             gdf_projected = gdf

        # Handle cases where there are too few points for k=8
        if len(gdf_projected) <= 8:
            print(f"  Warning: Insufficient points ({len(gdf_projected)}) for KNN with k=8. Skipping weights calculation for {city_name}.")
            # Add to processed set to avoid re-attempting
            processed_cities_for_weights.add(city_name)
            continue

        # Calculate KNN weights matrix
        wq = libpysal.weights.KNN.from_dataframe(gdf_projected, k=8)
        # No transformation needed for saving, will transform before G_Local calculation

        # Define the path to save the weights matrix
        weights_file_path = os.path.join(output_dir, f"knn_weights_2019_2024_{city_name.replace(' ', '_')}.gal")

        # Save the weights matrix explicitly opening and closing the file
        wfile = libpysal.io.open(weights_file_path, 'w')
        try:
            wfile.write(wq)
        finally:
            wfile.close()


        print(f"  KNN weights matrix saved for {city_name}: {weights_file_path}")

        processed_cities_for_weights.add(city_name)

    except Exception as e:
        print(f"  Error calculating or saving weights for {city_name}: {e}")
    finally:
        # Explicitly delete the GeoDataFrame and weights matrix
        if 'gdf' in locals():
            del gdf
        if 'gdf_projected' in locals():
            del gdf_projected
        if 'wq' in locals():
            del wq
        gc.collect()
        print(f"  Finished weights processing for {city_name}. Memory freed.")

print("\n--- Step 1 completed. KNN weights matrices saved. ---")